# Analyze a real public dataset · Solutions

Pandas, Clearly · Pandas for Data Science · Version 1.0.0

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates

Read, predict, run, change, and explain. Every worked example starts from fresh teaching data. The original student/café tables are fictional; the public Penguins files retain their documented source.

Run the setup cell first. Keep this notebook inside the downloaded course folder so its `data/` folder can be found. Exports go to `outputs/`. For public-data lessons, extract the full notebook practice pack.

Questions: What does one row represent? Which files are inputs, and where will your outputs be written?

In [ ]:
# Pandas, Clearly
# Dr. Ali Al Bataineh
# Computer Information Science
# Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates
# Run once before the lessons. It prepares explicitly fictional practice data.
# Questions: What does 'as pd' mean? Where will exported files be saved?
# Try: print(pd.__version__) and print(OUTPUT_DIR.resolve()).
# Explain: Why should source data and analysis outputs use different folders?
import os
import tempfile
from pathlib import Path
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "pandas-clearly-mpl"))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import sqlite3
from pathlib import Path
from io import StringIO

# Find data from the repository root or either notebook subdirectory.
# Browser workers mount the same data tree at /course/data.
COURSE_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents]
     if (parent / "data" / "students.csv").is_file()),
    Path.cwd(),
)
DATA_DIR = COURSE_ROOT / "data"
OUTPUT_DIR = COURSE_ROOT / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

_STUDENT_ROWS = [
    ["S001", "Amira", "RAK", "AI", 20, 8, 88, 95, "2026-09-01"],
    ["S002", "Bilal", "Dubai", "IT", 21, 5, 72, 85, "2026-09-02"],
    ["S003", "Chloe", "RAK", "AI", 19, 9, 91, 98, "2026-09-03"],
    ["S004", "Dana", "Dubai", "IT", 22, 4, 65, 78, "2026-09-04"],
    ["S005", "Ethan", "RAK", "IT", 20, 6, 79, 90, "2026-09-05"],
    ["S006", "Fatima", "Dubai", "AI", 21, 10, 95, 99, "2026-09-06"],
    ["S007", "Grace", "RAK", "AI", 23, 7, 84, 92, "2026-09-07"],
    ["S008", "Hassan", "Dubai", "IT", 20, 3, 58, 70, "2026-09-08"],
    ["S009", "Ivy", "RAK", "IT", 22, 6, 76, 88, "2026-09-09"],
    ["S010", "Jamal", "Dubai", "AI", 19, 8, 89, 96, "2026-09-10"],
    ["S011", "Khalid", "RAK", "AI", 24, 5, 68, 82, "2026-09-11"],
    ["S012", "Lina", "Dubai", "IT", 21, 7, 82, 94, "2026-09-12"],
]
_STUDENT_COLUMNS = ["Student_ID", "Name", "Campus", "Program", "Age",
                    "Study_Hours", "Score", "Attendance", "Join_Date"]

def students():
    """Return a new, clean practice DataFrame, never real student records."""
    return pd.DataFrame(_STUDENT_ROWS, columns=_STUDENT_COLUMNS)

def records():
    """Return the same practice records as a list of dictionaries."""
    return students().to_dict(orient="records")

def raw_students():
    """Return a fresh table with deliberate data-quality problems."""
    df = students()
    df["Score"] = df["Score"].astype(object)
    df["Study_Hours"] = df["Study_Hours"].astype(object)
    df.loc[0, "Name"] = " Amira "
    df.loc[1, "Campus"] = "dubai"
    df.loc[3, "Score"] = None
    df.loc[5, "Study_Hours"] = None
    df.loc[7, "Score"] = "absent"
    df.loc[10, "Attendance"] = 105
    return pd.concat([df, df.iloc[[2]]], ignore_index=True)

def departments():
    return pd.DataFrame({
        "Program": ["AI", "IT"],
        "Department": ["Computing", "Information Systems"],
        "Advisor": ["Dr. Noor", "Dr. Sami"],
    })

def assessment():
    return pd.DataFrame({
        "Student_ID": ["S001", "S002", "S003", "S004", "S005", "S006",
                       "S007", "S008", "S009", "S010", "S011", "S013"],
        "Quiz": [17, 14, 19, 13, 16, 19, 17, 11, 15, 18, 14, 16],
        "Project": [44, 36, 46, 34, 40, 47, 42, 30, 38, 45, 35, 41],
    })

def attendance():
    return pd.DataFrame({
        "Date": ["2026-10-01", "2026-10-02", "2026-10-03",
                 "2026-10-05", "2026-10-06", "2026-10-07"],
        "Visitors": [20, 24, 18, 30, 28, 32],
        "Campus": ["RAK"] * 6,
    })

# Create missing practice files. Existing files are preserved.
for _name, _frame in [
    ("students.csv", students()), ("raw_students.csv", raw_students()),
    ("departments.csv", departments()), ("assessment.csv", assessment()),
    ("attendance.csv", attendance()),
]:
    if not (DATA_DIR / _name).exists():
        _frame.to_csv(DATA_DIR / _name, index=False)
if not (DATA_DIR / "students.json").exists():
    students().to_json(DATA_DIR / "students.json", orient="records", indent=2)
if not (DATA_DIR / "students.xlsx").exists():
    try:
        students().to_excel(DATA_DIR / "students.xlsx", sheet_name="Students", index=False)
    except ImportError:
        print("Excel needs openpyxl. Install it before Module 20's Excel cell.")

pd.set_option("display.max_rows", 15)
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 110)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 110})
print("Ready. pandas", pd.__version__)
print("Practice data: 12 fictional students. Every helper returns a fresh table.")
print("Exports go to:", OUTPUT_DIR.resolve())


## Module 29. Analyze a real public dataset

Produce a traceable descriptive analysis with justified exclusions and visible denominators.

Before you start: You can clean, group, plot, and export data; review the public-data README and field dictionary.

Official reference: https://allisonhorst.github.io/palmerpenguins/reference/penguins.html

### M29C01. Check the source and audit missingness

Begin a public-data analysis by identifying the source, version, and field definitions. A checksum can confirm that the local source file matches the recorded bytes, while a missingness audit tells you what is available to analyze. These checks do not prove that every scientific measurement is correct.

**Read:** What does the checksum comparison establish?

Answer: The local file has the exact bytes recorded in the provenance manifest, assuming the manifest itself is trusted.

**Predict:** Which field has more missing values: sex or body_mass_g?

Answer: sex has 11 missing entries, compared with two missing body masses.

**Explain:** Why does file identity not prove scientific validity?

Answer: A file can be an exact copy and still contain measurement limitations, sampling constraints, or source errors.



In [ ]:
# Predict before running: Which field has more missing values: sex or body_mass_g?
import hashlib
folder = DATA_DIR / "public"
metadata = json.loads((folder / "provenance.json").read_text(encoding="utf-8"))
digest = hashlib.sha256((folder / "penguins.csv").read_bytes()).hexdigest()
assert digest == metadata["artifacts"]["penguins.csv"]["sha256"]
df = pd.read_csv(folder / "penguins.csv", na_values=["NA"])
result = pd.DataFrame({"Missing": df.isna().sum(), "Known": df.notna().sum()})
print("Dataset:", metadata["dataset"])
print("Source commit:", metadata["upstream"]["commit"])
print("Shape:", df.shape)
print(result)

**Your turn:** Read the source into df. Store a dictionary named result with keys total_rows, rows_with_any_missing, complete_rows, and rows_with_known_mass. Calculate those four counts across the source fields without deleting rows from df.

For the check, keep these result names: `df`, `result`.

**Hint:** Use isna().any(axis=1), dropna() only for counting, and body_mass_g.notna().sum().

**Watch for this:** A row position is a way to trace the source file, not an animal identifier.

In [ ]:
# Change this code to complete the task.
# Question: Read the source into df. Store a dictionary named result with keys total_rows, rows_with_any_missing, complete_rows, and rows_with_known_mass. Calculate those four counts across the source fields without deleting rows from df.
df = pd.read_csv(DATA_DIR / "public" / "penguins.csv", na_values=["NA"])
result = {
    "total_rows": len(df),
    "rows_with_any_missing": int(df.isna().any(axis=1).sum()),
    "complete_rows": len(df.dropna()),
    "rows_with_known_mass": int(df["body_mass_g"].notna().sum()),
}
print(result)
print("The original DataFrame still has", len(df), "rows.")

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result == {"total_rows": 344, "rows_with_any_missing": 11, "complete_rows": 333, "rows_with_known_mass": 342}
assert len(df) == 344
print("Code checks passed. Review the written explanation as well.")

There are 344 rows, 11 with at least one missing field, 333 complete rows, and 342 with known body mass. Complete-case filtering would discard nine additional known body masses because another field is missing.

**Quick check:** Which statement is justified by a matching recorded checksum?

1. The research has no limitations
2. The local source bytes match the recorded file
3. Every missing value is random

Answer: 2. A checksum establishes file identity, not representativeness or correctness of the study design.

**Explain what you learned:** Document source identity and data availability before reporting results.

### M29C02. Choose exclusions to match the question

A body-mass summary needs a known body mass and a grouping label. It does not automatically need recorded sex. Keep an exclusion audit and retain original units; this exercise does not replace missing measurements with invented values.

**Read:** Which two fields control inclusion in this cell?

Answer: species and body_mass_g.

**Predict:** Will every retained row also have a recorded sex?

Answer: No. Nine retained rows have a known mass but missing sex.

**Explain:** Why is source_row useful, and what does it not establish?

Answer: It traces a record to its one-based position in the source table. It is not a biological identity or a repeated-measurement ID.



In [ ]:
# Predict before running: Will every retained row also have a recorded sex?
df = pd.read_csv(DATA_DIR / "public" / "penguins.csv", na_values=["NA"])
df.insert(0, "source_row", np.arange(1, len(df) + 1))
required = ["species", "body_mass_g"]
missing_required = df[required].isna().any(axis=1)
audit = df.loc[missing_required, ["source_row"] + required].copy()
result = df.loc[~missing_required].copy()
assert result["body_mass_g"].gt(0).all()
print("Rows retained:", len(result))
print("Excluded rows:")
print(audit)
print("Retained rows with unrecorded sex:", result["sex"].isna().sum())

**Your turn:** Change the question to body-mass summaries by species and sex. Require species, sex, and body_mass_g. Recreate the retained result and exclusion audit from the original file; do not reuse the earlier filtered table.

For the check, keep these result names: `audit`, `result`.

**Hint:** Add sex to required and rebuild the missing-required mask.

**Watch for this:** Report exclusions and consider whether the remaining observations differ systematically; these checks do not establish random missingness.

In [ ]:
# Change this code to complete the task.
# Question: Change the question to body-mass summaries by species and sex. Require species, sex, and body_mass_g. Recreate the retained result and exclusion audit from the original file; do not reuse the earlier filtered table.
df = pd.read_csv(DATA_DIR / "public" / "penguins.csv", na_values=["NA"])
df.insert(0, "source_row", np.arange(1, len(df) + 1))
required = ["species", "sex", "body_mass_g"]
missing_required = df[required].isna().any(axis=1)
audit = df.loc[missing_required, ["source_row"] + required].copy()
result = df.loc[~missing_required].copy()
print("Rows retained:", len(result))
print("Rows excluded:", len(audit))
print(audit)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert len(result) == 333
assert len(audit) == 11
assert result[["species", "sex", "body_mass_g"]].notna().all().all()
assert len(result) + len(audit) == 344
print("Code checks passed. Review the written explanation as well.")

The revised question retains 333 records and excludes 11. The decision changed because the requested grouping now needs sex; it is not a rule that every analysis must use complete cases.

**Quick check:** Why keep some rows with missing sex for the original mass-by-species question?

1. Sex is not required for that stated summary
2. Missing sex is equivalent to male
3. Missing measurements never matter

Answer: 1. A required-field policy should match the analysis question and still report what is missing.

**Explain what you learned:** Define the required fields and preserve an exclusion audit.

### M29C03. Report group means with their denominators

A group mean uses the known measurements in that group. Show total observations, known measurements, and missing measurements beside the mean so readers can see its denominator. The summaries describe these observations; they do not establish causal species or island effects.

**Read:** What is the denominator for the Adelie mean mass?

Answer: 151 known body masses, not all 152 Adelie observations.

**Predict:** Which species has no missing body masses in this table?

Answer: Chinstrap: all 68 observations have known body mass.

**Explain:** Can these group means alone explain why the species differ in body mass?

Answer: No. They are descriptive summaries; causal explanations require an appropriate design and relevant evidence.



In [ ]:
# Predict before running: Which species has no missing body masses in this table?
df = pd.read_csv(DATA_DIR / "public" / "penguins.csv", na_values=["NA"])
result = df.groupby("species", as_index=False).agg(Observations=("species", "size"), Known_mass=("body_mass_g", "count"), Mean_mass_g=("body_mass_g", "mean"))
result["Missing_mass"] = result["Observations"] - result["Known_mass"]
print(result.round({"Mean_mass_g": 2}))
print("Total observed records:", result["Observations"].sum())
print("Total known body masses:", result["Known_mass"].sum())

**Your turn:** Group by island instead of species. Store result with island as a normal column, sorted in ascending island order, and report Observations, Known_mass, Missing_mass, and Mean_mass_g. Reconcile the total observations and known measurements with the full table.

For the check, keep these result names: `result`.

**Hint:** Change the grouping key and keep the matching size/count denominators.

**Watch for this:** Sample sizes and species composition differ across islands. A descriptive comparison is not an adjusted causal analysis.

In [ ]:
# Change this code to complete the task.
# Question: Group by island instead of species. Store result with island as a normal column, sorted in ascending island order, and report Observations, Known_mass, Missing_mass, and Mean_mass_g. Reconcile the total observations and known measurements with the full table.
df = pd.read_csv(DATA_DIR / "public" / "penguins.csv", na_values=["NA"])
result = df.groupby("island", as_index=False).agg(Observations=("island", "size"), Known_mass=("body_mass_g", "count"), Mean_mass_g=("body_mass_g", "mean"))
result["Missing_mass"] = result["Observations"] - result["Known_mass"]
assert result["Observations"].sum() == len(df)
assert result["Known_mass"].sum() == df["body_mass_g"].count()
print(result.round({"Mean_mass_g": 2}))

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result["Observations"].tolist() == [168, 124, 52]
assert result["Known_mass"].tolist() == [167, 124, 51]
assert result["Missing_mass"].sum() == 2
assert np.isclose(result.set_index("island").loc["Dream", "Mean_mass_g"], 460400 / 124)
print("Code checks passed. Review the written explanation as well.")

Biscoe has 168 observations and 167 known masses; Dream has 124 and 124; Torgersen has 52 and 51. Their mean masses are approximately 4,716.02 g, 3,712.90 g, and 3,706.37 g. Different species composition is one reason to avoid interpreting these as an isolated island effect.

**Quick check:** What should accompany a mean when values are missing?

1. Only the largest value
2. The number of known measurements used
3. A claim that missing values are zero

Answer: 2. The known-value count tells the reader what supports the mean.

**Explain what you learned:** Pair every summary with its relevant count and interpretation.

### M29C04. Create a chart and an auditable export

Combine the source, missingness policy, grouped summary, chart, and export into one repeatable cell. Keep counts in the exported table and label the plotted units. A chart can summarize the sampled measurements without claiming a population estimate or a causal finding.

**Read:** What does each n label above a bar count?

Answer: The known body masses used to calculate that species mean.

**Predict:** Will converting grams to kilograms change which species has the largest mean in this table?

Answer: No. Every mean is divided by the same positive factor of 1,000.

**Explain:** Why export the counts and missing counts as well as the plotted mean?

Answer: They preserve the analysis denominator and make it easier to audit how the result was calculated.



In [ ]:
# Predict before running: Will converting grams to kilograms change which species has the largest mean in this table?
df = pd.read_csv(DATA_DIR / "public" / "penguins.csv", na_values=["NA"])
result = df.groupby("species", as_index=False).agg(Observations=("species", "size"), Known_mass=("body_mass_g", "count"), Mean_mass_g=("body_mass_g", "mean"))
result["Missing_mass"] = result["Observations"] - result["Known_mass"]
result["Mean_mass_kg"] = result["Mean_mass_g"] / 1000
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(result["species"], result["Mean_mass_kg"], color="#326f79")
for i, row in result.iterrows():
    ax.text(i, row["Mean_mass_kg"] + 0.08, f"n={int(row['Known_mass'])}", ha="center")
ax.set(title="Mean observed penguin body mass, 2007–2009", xlabel="Species", ylabel="Mean body mass (kg)", ylim=(0, 6))
fig.tight_layout()
csv_path = OUTPUT_DIR / "public_penguins_mass_summary.csv"
plot_path = OUTPUT_DIR / "public_penguins_mass_summary.png"
result.to_csv(csv_path, index=False)
fig.savefig(plot_path, dpi=160)
print(result.round({"Mean_mass_g": 2, "Mean_mass_kg": 3}))
print("Source: Palmer Penguins; DOI 10.5281/zenodo.3960218. Means use known body masses.")
print("These summaries describe sampled observations, not causal effects or a population census.")
plt.show()

**Your turn:** Build result as a summary by species in ascending species order, with columns species, Observations, Known_length, Median_flipper_mm, and Missing_length. Plot the medians with known-count labels on axes named ax, using the y-axis label "Median flipper length (mm)". Set csv_path and plot_path to public_penguins_flipper_summary.csv and public_penguins_flipper_summary.png in OUTPUT_DIR, then export the table and chart.

For the check, keep these result names: `ax`, `csv_path`, `plot_path`, `result`.

**Hint:** Change the measurement, summary method, labels, and filenames together; do not reuse the mass units.

**Watch for this:** Do not imply uncertainty intervals or representativeness that the chart has not calculated or established.

In [ ]:
# Change this code to complete the task.
# Question: Build result as a summary by species in ascending species order, with columns species, Observations, Known_length, Median_flipper_mm, and Missing_length. Plot the medians with known-count labels on axes named ax, using the y-axis label "Median flipper length (mm)". Set csv_path and plot_path to public_penguins_flipper_summary.csv and public_penguins_flipper_summary.png in OUTPUT_DIR, then export the table and chart.
df = pd.read_csv(DATA_DIR / "public" / "penguins.csv", na_values=["NA"])
result = df.groupby("species", as_index=False).agg(Observations=("species", "size"), Known_length=("flipper_length_mm", "count"), Median_flipper_mm=("flipper_length_mm", "median"))
result["Missing_length"] = result["Observations"] - result["Known_length"]
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(result["species"], result["Median_flipper_mm"], color="#326f79")
for i, row in result.iterrows():
    ax.text(i, row["Median_flipper_mm"] + 3, f"n={int(row['Known_length'])}", ha="center")
ax.set(title="Median observed penguin flipper length, 2007–2009", xlabel="Species", ylabel="Median flipper length (mm)", ylim=(0, 250))
fig.tight_layout()
csv_path = OUTPUT_DIR / "public_penguins_flipper_summary.csv"
plot_path = OUTPUT_DIR / "public_penguins_flipper_summary.png"
result.to_csv(csv_path, index=False)
fig.savefig(plot_path, dpi=160)
print(result)
print("Source: Palmer Penguins; DOI 10.5281/zenodo.3960218. Medians use known lengths.")
print("The chart describes the available study observations; it is not a causal analysis.")
plt.show()

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result["Median_flipper_mm"].tolist() == [190.0, 196.0, 216.0]
assert result["Known_length"].tolist() == [151, 68, 123]
assert result["Missing_length"].sum() == 2
assert csv_path.exists() and plot_path.exists()
assert ax.get_ylabel() == "Median flipper length (mm)"
assert len(pd.read_csv(csv_path)) == 3
print("Code checks passed. Review the written explanation as well.")

The median flipper lengths are 190 mm for Adelie, 196 mm for Chinstrap, and 216 mm for Gentoo, based on 151, 68, and 123 known measurements. The table and chart use the same units and denominators.

**Quick check:** Which label correctly describes the plotted output?

1. The true body mass of every penguin worldwide
2. Mean observed body mass in the supplied study records
3. Proof that island location causes body mass

Answer: 2. The chart summarizes the available observations with their measurement counts.

**Explain what you learned:** Deliver the result with its source, units, counts, and limits.

## Keep practicing

Can you apply this idea to a different table without copying the example?

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates